# Lesson 10 - AI-agentit tuotannossa

Tässä oppitunnissa opit **tuotantokuvioita** AI-agenttien käyttämiseen Microsoft Agent Frameworkilla (Python). Käsittelemme:

- **Havaittavuus** — ajoituksen ja lokituksen lisääminen agentin vuorovaikutuksiin
- **Arviointi** — arviointianturin käyttäminen vastausten laadun pisteyttämiseen
- **Kustannusten hallinta** — strategioita tokenien optimointiin ja mallin valintaan

Skenaario on **matkatoimisto**, joka auttaa käyttäjiä suunnittelemaan matkoja, ja jonka päälle on rakennettu valvonta ja arviointi.


## Asennus


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Tuotantoon liittyvät seikat

Tekoälyagenttien siirtäminen prototyypeistä tuotantoon vaatii huolellista huomiota kolmeen pilariseen:

1. **Havaittavuus** — Sinun on nähtävä, mitä agentti tekee, kuinka kauan se kestää ja mitä työkaluja se kutsuu. Ilman jäljitystä ja lokitusta tuotannon virheiden vianmääritys on lähes mahdotonta.

2. **Arviointi** — Automaattiset laadunvarmistukset varmistavat, että agentin vastaukset pysyvät ajan mittaan tarkkoina, täydellisinä ja hyödyllisinä. Arvioija-agentti voi pisteyttää vastauksia määriteltyjä kriteerejä vasten.

3. **Kustannusten hallinta** — Tokenien käyttö vaikuttaa suoraan kustannuksiin. Strategiat, kuten kehotteen optimointi, mallin valinta ja välimuistin käyttö, auttavat pitämään kulut kurissa ilman, että laatu kärsii.


## Havainnoitavan agentin rakentaminen

Määrittelemme matkustustyökalut ja kääritään agenttikutsu ajoituksen ympärille, jotta voimme seurata viivettä. Tuotannossa integroisit tämän OpenTelemetryn tai vastaavan jäljitystaustan kanssa.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Arviointimallit

Yleinen tuotantotapa on käyttää toista agenttia **arvioijana**. Arvioija pisteyttää ensisijaisen agentin vastauksen ennalta määriteltyjen kriteerien, kuten täydellisyyden, tarkkuuden ja hyödyllisyyden, perusteella.

Tämä mahdollistaa:
- Automaattiset laatukriteerit ennen kuin vastaukset saavuttavat käyttäjät
- Regressioiden tunnistamisen, kun kehotteet tai mallit muuttuvat
- Agentin suorituskyvyn jatkuvan seurannan ajan kuluessa


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Kustannustenhallintastrategiat

Kustannusten hallinta on kriittistä tuotannon tekoälyagenttien kanssa. Tässä ovat keskeiset strategiat:

| Strategia | Kuvaus |
|---|---|
| **Kehoitteiden optimointi** | Pidä järjestelmäohjeet tiiviinä. Poista päällekkäinen konteksti vähentääksesi syötteen tokeneita. |
| **Mallin valinta** | Käytä pienempiä, edullisempia malleja (esim. GPT-4o-mini) yksinkertaisiin tehtäviin kuten luokitteluun tai tietojen poimintaan, ja varaa suuremmat mallit monimutkaiseen päättelyyn. |
| **Välimuistitus** | Välimuistita työkalujen tulokset ja usein toistuvat kyselyt välttääksesi turhia API-kutsuja. |
| **Token-budjetit** | Aseta `max_tokens` rajat estääksesi odottamattoman pitkiä vastauksia. |
| **Ryhmittely** | Ryhmittele useita käyttäjäkyselyjä yhdeksi API-kutsuksi, missä se on mahdollista. |

Käytännössä kerroksittainen lähestymistapa toimii hyvin: ohjaa suoraviivaiset pyynnöt nopealle, edulliselle mallille ja nosta monimutkaiset kyselyt vain kykenevämmälle mallille.


## Yhteenveto

Tässä oppitunnissa opit miten:

1. **Lisätä havainnoitavuutta** agenttien vuorovaikutukseen aikaleimojen ja lokituksen avulla, luoden perustan jäljittämiselle ja valvonnalle.
2. **Arvioida agenttien vastauksia** automaattisesti arvioija-agentin avulla, joka pisteyttää täydellisyyden, tarkkuuden ja hyödyllisyyden.
3. **Hallita kustannuksia** kehotteiden optimoinnilla, mallin valinnalla, välimuistilla ja token-budjeteilla.

Nämä tuotantomallit auttavat varmistamaan, että tekoälyagenttisi ovat luotettavia, mitattavissa ja kustannustehokkaita suuressa mittakaavassa.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
